# Temperature — 질문할 때마다 같은 답을 받기 (Gemini)

`1_1_openai` 에서 같은 질문을 두 번 보내 보면 **답이 매번 조금씩 다릅니다.**
이 노트북은 그 이유와, `temperature` 를 **0 ~ 1 사이**에서 조절해 **매번 같은 답을 받는** 방법을 다룹니다.

모델은 OpenAI 대신 **Google Gemini** 를 씁니다.
Gemini 는 **OpenAI 와 같은 형식의 주소(`GEMINI_URL`)** 를 제공하므로,
`1_1_openai` 에서 배운 `OpenAI` 클라이언트를 **주소와 키만 바꿔** 그대로 씁니다.

## 이 노트북의 흐름

| 단계 | 내용 |
|---|---|
| **Step 1~2** | 준비 — 라이브러리, API 키, Gemini 주소 |
| **Step 3** | temperature 가 무엇인가 |
| **Step 4** | 같은 질문을 여러 번 보내는 함수 만들기 |
| **Step 5** | `temperature=1` — 답이 매번 달라진다 |
| **Step 6** | `temperature=0` — 답이 비슷해진다 |
| **Step 7** | `top_p=0` 추가 — 1등 후보만 남긴다 |
| **Step 8** | 답을 짧게 — **매번 똑같은 답** |
| **Step 9** | `temperature=0.3` 으로도 같은 답이 나오는지 확인 |

> **이 노트북에서 기억할 한 가지**
> 같은 답을 받으려면 temperature 하나만으로는 부족합니다.
> **`temperature` 낮게 + `top_p=0` + 짧은 답** 세 가지를 함께 씁니다.

## Step 1. 준비물

`1_1_openai` 의 Step 1 에 `time` 하나가 추가됐습니다.

| 가져오는 것 | 하는 일 |
|---|---|
| `os` | 환경변수를 읽는다 |
| `time` | 호출 사이에 잠깐 쉰다 (Step 4 에서 설명) |
| `dotenv` | `.env` 파일을 환경변수로 올려 준다 |
| `OpenAI` | OpenAI 공식 클라이언트 — **Gemini 에도 그대로 쓴다** |

In [1]:
import os
import time
from dotenv import load_dotenv
from openai import OpenAI

## Step 2. Gemini 키와 주소로 클라이언트 만들기

`.env` 에 두 값이 있어야 합니다.

```
GOOGLE_API_KEY=AIza...
GEMINI_URL=https://generativelanguage.googleapis.com/v1beta/openai/
```

`1_1_openai` 와 달라진 곳은 **클라이언트를 만드는 한 줄**뿐입니다.

| | `1_1_openai` | 이 노트북 |
|---|---|---|
| 키 | `OPENAI_API_KEY` (자동으로 찾음) | `GOOGLE_API_KEY` (직접 넘김) |
| 주소 | OpenAI 기본 주소 (생략) | `base_url=GEMINI_URL` |

```python
openai = OpenAI(base_url=gemini_url, api_key=google_api_key)
```

이렇게 만들면 이후의 `openai.chat.completions.create(...)` 는 **OpenAI 가 아니라 Gemini 로** 갑니다.

In [2]:
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')
gemini_url = os.getenv('GEMINI_URL')

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:6]}")
else:
    print("Google API Key not set")

print(f"Gemini URL: {gemini_url}")

openai = OpenAI(base_url=gemini_url, api_key=google_api_key)

Google API Key exists and begins AQ.Ab8
Gemini URL: https://generativelanguage.googleapis.com/v1beta/openai/


## Step 3. temperature 가 무엇인가

LLM 은 답을 **한 단어(토큰)씩** 만듭니다. 매 순간 다음 단어 후보마다 확률이 있습니다.

```
"가장 먼저 챙겨야 할 보안 이슈는 ___"

  다중 인증   45%
  랜섬웨어    30%
  공급망      15%
  ...
```

**temperature 는 이 확률 중에서 얼마나 '모험'을 할지 정하는 값**입니다.

| temperature | 고르는 방식 | 결과 |
|---|---|---|
| `0` | 1등 단어를 고름 | 답이 거의 같아짐 |
| `0.3` ~ `0.5` | 1등 위주, 가끔 2~3등 | 조금씩 다름 |
| `1` (기본값) | 확률대로 뽑기 | 매번 다름 |

### 함께 쓰는 값: `top_p`

| 값 | 뜻 | 같은 답을 원할 때 |
|---|---|---|
| `top_p` | 확률 상위 몇 % 후보 안에서만 고를지 (기본 1 = 전체) | `0` → **1등 후보만** 남김 |

> **OpenAI 와 다른 점 — `seed` 가 없습니다**
> OpenAI 에서는 `seed=42` 처럼 난수 시드를 고정할 수 있지만,
> Gemini 의 OpenAI 호환 주소는 `seed` 를 받지 않고 `Unknown name "seed"` 에러를 냅니다.
> 그래서 이 노트북은 `temperature` 와 `top_p` 만 씁니다.

> **모델 고르기 — 모든 Gemini 모델이 temperature 를 잘 따르지는 않습니다**
> 이 노트북을 만들 때 `temperature=0`, `top_p=0` 으로 5번씩 물어본 결과입니다.
>
> | 모델 | 5번 중 서로 다른 답 |
> |---|---|
> | `gemini-3.1-flash-lite` | **1개** (모두 같음) |
> | `gemini-3-flash-preview` | 2개 |
> | `gemini-3.5-flash-lite` | 5개 (모두 다름) |
>
> 그래서 `gemini-3.1-flash-lite` 를 씁니다.

## Step 4. 같은 질문을 여러 번 보내는 함수

비교하려면 **같은 질문을 여러 번** 보내야 하므로 함수로 만들어 둡니다.

```python
ask(question, temperature, top_p=None)
```

| 인자 | 뜻 |
|---|---|
| `question` | 보낼 질문 |
| `temperature` | 이번 호출의 temperature (0 ~ 1) |
| `top_p` | 후보 범위. `None` 이면 보내지 않음 (Step 7 부터 사용) |

`ask_n_times()` 는 같은 질문을 `n` 번 보내고, 마지막에 **서로 다른 답이 몇 개인지** 세어 줍니다.
**1개면 n 번 모두 똑같은 답**이라는 뜻입니다.

### 왜 `time.sleep(4.5)` 를 넣었나

Gemini **무료 등급은 모델마다 1분에 15번**까지만 호출할 수 있습니다.
넘으면 `429 RESOURCE_EXHAUSTED` 에러가 납니다.
60초 ÷ 15번 = 4초이므로, 여유 있게 **호출할 때마다 4.5초씩 쉬게** 했습니다.
그래서 셀 하나가 실행되는 데 수십 초가 걸립니다.

In [3]:
MODEL = "gemini-3.1-flash-lite"

system_prompt = """
당신은 IT 보안 전문가입니다.
시스템 보안에 대한 전문 지식을 제공하며 새롭게 발생하는 보안 문제에 대해서도 대처 방법을 제공합니다
"""

question = "나는 IT 회사의 보안 책임자입니다. 지금 가장 먼저 챙겨야 할 보안 이슈 하나를 두 문장으로 알려주세요."


def ask(question, temperature, top_p=None):
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": question}]

    # top_p 는 값을 줬을 때만 보냅니다.
    options = {}
    if top_p is not None:
        options["top_p"] = top_p

    time.sleep(4.5)  # 무료 등급 제한(1분 15번)을 넘지 않도록

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=500,
        **options,
    )
    return response.choices[0].message.content


def ask_n_times(question, temperature, n=3, top_p=None):
    answers = []
    for i in range(n):
        answer = ask(question, temperature, top_p)
        answers.append(answer)
        print(f"[{i + 1}번째] {answer}\n")

    # set() 은 중복을 없애므로, 길이가 1 이면 n 번 모두 같은 답이라는 뜻
    print(f"→ {n}번 중 서로 다른 답: {len(set(answers))}개")
    return answers

## Step 5. `temperature=1` — 답이 매번 달라진다

먼저 비교 기준으로 기본값인 `temperature=1` 로 세 번 물어봅니다.
**같은 질문, 같은 system 프롬프트, 같은 모델**인데도 답이 달라지는 것을 보세요.

In [4]:
answers_t1 = ask_n_times(question, temperature=1)

[1번째] 현재 가장 시급한 이슈는 **AI 도구 사용으로 인한 기업 내부 데이터 유출 및 공급망 공격**입니다. 임직원이 업무에 사용하는 생성형 AI에 민감 정보를 입력하지 않도록 가이드라인을 수립하고, AI 관련 라이브러리 및 외부 오픈소스의 보안 검증 체계를 즉시 강화하십시오.

[2번째] 현재 가장 시급한 이슈는 생성형 AI 도입에 따른 기업 데이터 유출 및 공급망 공격을 통한 권한 탈취입니다. 임직원의 AI 사용 가이드를 즉시 수립하고, 모든 시스템 접속 단계에서 제로 트러스트(Zero Trust) 기반의 다중 인증과 최소 권한 원칙을 강제하여 방어 체계를 재정비하십시오.

[3번째] 현재 가장 시급한 보안 이슈는 **AI 도구 사용으로 인한 기업 데이터 유출 및 공급망 공격(Supply Chain Attack)**입니다. 임직원들이 사내 기밀을 외부 AI 모델에 입력하지 않도록 가이드라인을 수립하고, 동시에 소프트웨어 개발 단계에서 의존성 라이브러리의 무결성을 검증하는 보안 체계를 즉시 강화하십시오.

→ 3번 중 서로 다른 답: 3개


## Step 6. `temperature=0` — 답이 비슷해진다

이번에는 `temperature=0` 으로 세 번 물어봅니다.
Step 5 와 비교해 **답의 내용과 첫 문장이 얼마나 비슷해졌는지** 보세요.

> 1등 단어를 고르라고 했는데도 가끔 달라지는 이유
> 서버는 여러 요청을 묶어 GPU 에서 병렬로 계산합니다. 이때 생기는 아주 작은 계산 오차 때문에
> 1등과 2등 확률이 거의 비슷한 지점에서 **가끔 2등이 1등으로 뒤집힙니다.**
> 한 단어가 바뀌면 그 뒤 문장 전체가 달라집니다.

In [5]:
answers_t0 = ask_n_times(question, temperature=0)

[1번째] 현재 가장 시급한 보안 이슈는 **AI 도구 도입에 따른 기업 내부 데이터 유출 및 프롬프트 인젝션 공격**입니다. 임직원들이 업무에 생성형 AI를 무분별하게 사용하지 않도록 데이터 보안 가이드라인을 즉시 수립하고, AI 서비스에 대한 접근 제어 및 모니터링 체계를 강화하십시오.

[2번째] 현재 가장 시급한 보안 이슈는 **AI 도구 도입에 따른 기업 내부 데이터 유출 및 프롬프트 인젝션 공격**입니다. 임직원들이 업무에 생성형 AI를 무분별하게 사용하지 않도록 데이터 보안 가이드라인을 즉시 수립하고, AI 서비스에 대한 접근 제어 및 모니터링 체계를 강화하십시오.

[3번째] 현재 가장 시급한 보안 이슈는 **AI 도구 도입에 따른 기업 내부 데이터 유출 및 프롬프트 인젝션 공격**입니다. 임직원들이 업무에 생성형 AI를 무분별하게 사용하지 않도록 데이터 보안 가이드라인을 즉시 수립하고, AI 서비스에 대한 접근 제어 및 모니터링 체계를 강화하십시오.

→ 3번 중 서로 다른 답: 1개


## Step 7. `top_p=0` 추가 — 1등 후보만 남긴다

Step 3 에서 본 `top_p` 를 0 으로 고정합니다.

```python
ask_n_times(question, temperature=0, top_p=0)
```

`top_p=0` 은 후보를 **1등 하나만** 남깁니다.
다만 **답이 두 문장으로 길면** 중간에 한 단어가 뒤집힐 기회가 많아, 여전히 달라질 수 있습니다.

In [6]:
answers_top_p = ask_n_times(question, temperature=0, top_p=0)

[1번째] 현재 가장 시급한 보안 이슈는 **AI 도구 도입에 따른 기업 내부 데이터 유출 및 프롬프트 인젝션 공격**입니다. 임직원들이 업무에 생성형 AI를 무분별하게 사용하지 않도록 데이터 보안 가이드라인을 즉시 수립하고, AI 서비스에 대한 접근 제어 및 모니터링 체계를 강화하십시오.

[2번째] 현재 가장 시급한 보안 이슈는 **AI 도구 도입에 따른 기업 내부 데이터 유출 및 프롬프트 인젝션 공격**입니다. 임직원들이 업무에 생성형 AI를 무분별하게 사용하지 않도록 데이터 보안 가이드라인을 즉시 수립하고, AI 서비스에 대한 접근 제어 및 모니터링 체계를 강화하십시오.

[3번째] 현재 가장 시급한 보안 이슈는 **AI 도구 도입에 따른 기업 내부 데이터 유출 및 프롬프트 인젝션 공격**입니다. 임직원들이 업무에 생성형 AI를 무분별하게 사용하지 않도록 데이터 보안 가이드라인을 즉시 수립하고, AI 서비스에 대한 접근 제어 및 모니터링 체계를 강화하십시오.

→ 3번 중 서로 다른 답: 1개


## Step 8. 답을 짧게 — 매번 똑같은 답

마지막 열쇠는 **답의 길이**입니다.
단어가 적을수록 뒤집힐 지점이 줄어들어, 같은 답이 나올 가능성이 크게 올라갑니다.

질문 끝을 이렇게 바꿉니다.

```
두 문장으로 알려주세요   ->   한 문장(40자 이내)으로 알려주세요
```

이번에는 **5번** 물어봅니다.
`서로 다른 답: 1개` 가 나오면 5번 모두 똑같은 답입니다.

In [7]:
short_question = "나는 IT 회사의 보안 책임자입니다. 지금 가장 먼저 챙겨야 할 보안 이슈 하나를 한 문장(40자 이내)으로 알려주세요."

answers_short = ask_n_times(short_question, temperature=0, top_p=0, n=5)

[1번째] 공급망 공격을 대비한 소프트웨어 구성 요소(SBOM) 관리와 권한 통제를 강화하세요.

[2번째] 공급망 공격을 대비한 소프트웨어 구성 요소(SBOM) 관리와 권한 통제를 강화하세요.

[3번째] 공급망 공격을 대비한 소프트웨어 구성 요소(SBOM) 관리와 권한 통제를 강화하세요.

[4번째] 공급망 공격을 대비한 소프트웨어 구성 요소(SBOM) 관리와 권한 통제를 강화하세요.

[5번째] 공급망 공격을 대비한 소프트웨어 구성 요소(SBOM) 관리와 권한 통제를 강화하세요.

→ 5번 중 서로 다른 답: 1개


## Step 9. `temperature=0.3` 으로도 같은 답이 나오는지

0 과 1 사이의 다른 값으로도 같은 답이 나오는지 확인해 봅니다.

`top_p=0` 이면 후보가 **1등 하나만** 남으므로, temperature 를 0.3 으로 올려도 고를 수 있는 단어가 하나뿐입니다.
그래서 **Step 8 과 같은 답**이 나와야 합니다.

In [8]:
answers_t03 = ask_n_times(short_question, temperature=0.3, top_p=0, n=5)

print("Step 8 (temperature=0) 과 같은 답인가?", set(answers_t03) == set(answers_short))

[1번째] 공급망 공격을 대비한 소프트웨어 구성 요소(SBOM) 관리와 권한 통제를 강화하세요.

[2번째] 공급망 공격을 대비한 소프트웨어 구성 요소(SBOM) 관리와 권한 통제를 강화하세요.

[3번째] 공급망 공격을 대비한 소프트웨어 구성 요소(SBOM) 관리와 권한 통제를 강화하세요.

[4번째] 공급망 공격을 대비한 소프트웨어 구성 요소(SBOM) 관리와 권한 통제를 강화하세요.

[5번째] 공급망 공격을 대비한 소프트웨어 구성 요소(SBOM) 관리와 권한 통제를 강화하세요.

→ 5번 중 서로 다른 답: 1개
Step 8 (temperature=0) 과 같은 답인가? True


---

## 정리

| 설정 | 결과 |
|---|---|
| `temperature=1` | 매번 다름 |
| `temperature=0` | 내용이 비슷해짐 |
| `temperature=0` + `top_p=0` | 1등 후보만 남아 더 비슷해짐 |
| 위 설정 + **짧은 답** | **매번 똑같은 답** |
| `temperature=0.3` + `top_p=0` + 짧은 답 | `top_p=0` 이면 temperature 0 과 같은 답 |

### 같은 답을 받는 설정 (복사해서 쓰기)

```python
openai = OpenAI(base_url=os.getenv("GEMINI_URL"), api_key=os.getenv("GOOGLE_API_KEY"))

response = openai.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=messages,
    temperature=0,     # 0 ~ 0.3 정도로 낮게
    top_p=0,           # 1등 후보만
    max_tokens=500,
)
```

### 꼭 기억할 것

| | 내용 |
|---|---|
| **주소만 바꾸면 Gemini** | `OpenAI(base_url=GEMINI_URL, api_key=GOOGLE_API_KEY)` |
| **temperature 는 '모험 정도'** | 0 이면 1등 단어 위주, 1 이면 확률대로 뽑기 |
| **temperature 혼자로는 부족** | `top_p=0` 을 함께 쓰고, 답을 짧게 한다 |
| **Gemini 는 `seed` 가 없다** | OpenAI 호환 주소에서 `seed` 를 보내면 에러 |
| **모델마다 다르다** | 같은 설정이어도 `gemini-3.5-flash-lite` 는 매번 다른 답을 했다 |
| **100% 보장은 아님** | 서버 계산 오차로 가끔 다른 답이 섞일 수 있고, 모델이 업데이트되면 답이 바뀔 수 있다 |

### 직접 해 보기

1. Step 9 의 `temperature` 를 `0.7`, `1` 로 올려 보세요. `top_p=0` 이면 여전히 같은 답이 나오나요?
2. Step 8 에서 `top_p=0` 을 빼고 실행해 보세요. 서로 다른 답이 몇 개 나오나요?
3. Step 4 의 `MODEL` 을 `"gemini-3.5-flash-lite"` 로 바꾸고 Step 8 을 실행해 보세요. 결과가 어떻게 달라지나요?
4. `ask()` 안에 `seed=42` 를 넣어 보세요. 어떤 에러가 나나요?